# Feature Selection via ARIMA & LSTM Training
## Permutation Feature Importance on Electricity Demand Data

This notebook:
1. Loads and preprocesses `full_data_2016_onwards.csv`
2. Trains a **SARIMAX** (ARIMA with exogenous features) on an aggregated time series
3. Trains a **PyTorch LSTM** on the full multi-location dataset with GPU + AMP
4. Runs **permutation feature importance** on both models to rank features


In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────────
import os
import pickle
import random
import warnings
from pathlib import Path

# ── Numerical / data handling ────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Progress bars ─────────────────────────────────────────────────────────────────
from tqdm.auto import tqdm

# ── Scikit-learn utilities ────────────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ── Statsmodels – SARIMAX (ARIMA with exogenous variables) ───────────────────────
from statsmodels.tsa.statespace.sarimax import SARIMAX

# ── PyTorch ───────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
print("All libraries imported successfully.")


## Configuration
All tunable hyper-parameters and paths in one place.


In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────────
BASE_DIR       = Path("..").resolve()
DATA_PATH      = BASE_DIR / "data" / "full_data_2016_onwards.csv"
CHECKPOINT_DIR = BASE_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LSTM_CKPT  = CHECKPOINT_DIR / "lstm_checkpoint.pt"   # PyTorch checkpoint
ARIMA_CKPT = CHECKPOINT_DIR / "arima_checkpoint.pkl" # Pickled SARIMAXResults

# ── ARIMA / SARIMAX ────────────────────────────────────────────────────────────
# None = auto-select the location with the most rows; set a string to pin one.
ARIMA_LOCATION  = None
ARIMA_ORDER     = (1, 1, 1)   # (p, d, q) – kept simple for tractability
ARIMA_MAX_PTS   = 8_760       # cap training series at 1 year of hourly data
ARIMA_N_REPEATS = 5           # shuffles per feature for permutation importance

# ── LSTM ───────────────────────────────────────────────────────────────────────
SEQ_LEN       = 24    # look-back window in hours (1 day)
BATCH_SIZE    = 64    # small to respect the 8 GB shared VRAM budget
HIDDEN_SIZE   = 128   # LSTM hidden units per layer
NUM_LAYERS    = 2     # stacked LSTM depth
DROPOUT       = 0.2   # applied between LSTM layers (ignored when NUM_LAYERS=1)
EPOCHS        = 10    # total training epochs
LR            = 1e-3  # Adam learning rate
CLIP_GRAD     = 1.0   # gradient-norm clipping threshold
SAVE_EVERY    = 2     # save checkpoint every N epochs
LSTM_N_REPEATS = 5    # shuffles per feature for permutation importance

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"  GPU  : {props.name}")
    print(f"  VRAM : {props.total_memory / 1e9:.1f} GB")


## Data Loading & Inspection


In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Shape : {df.shape}")
print(f"\nDtypes :\n{df.dtypes.to_string()}")
print(f"\nFirst 2 rows :\n{df.head(2).to_string()}")
print(f"\nMissing values (non-zero only) :")
nulls = df.isnull().sum()
print(nulls[nulls > 0].to_string() if nulls.any() else "  None")
print(f"\nUnique locations : {df['location_id'].nunique()}")
print(f"Date range       : {df['timestamp'].min()}  →  {df['timestamp'].max()}")


## Preprocessing

Steps:
1. Parse `timestamp` → `datetime64`
2. Label-encode `location_id` (string → int); drop the redundant `location` string column
3. Derive the full feature list (all numeric columns except `y`)
4. Fit `StandardScaler` on features and target separately — scaling is important for both LSTM convergence and SARIMAX numerical stability


In [ ]:
# ── 1. Parse timestamp ────────────────────────────────────────────────────────
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Sort so that every per-location slice is chronologically ordered
df = df.sort_values(["location_id", "timestamp"]).reset_index(drop=True)

# ── 2. Encode / drop categoricals ─────────────────────────────────────────────
le = LabelEncoder()
# Add an integer-encoded location column; keep original for filtering later
df["location_id_enc"] = le.fit_transform(df["location_id"])

# 'location' (city string) duplicates information already in location_id/lat/lon
df = df.drop(columns=["location"])

# ── 3. Define feature columns ─────────────────────────────────────────────────
TARGET = "y"

# Exclude non-feature columns from the feature set
_EXCLUDE = {"timestamp", "location_id", TARGET}
FEATURE_COLS = [
    c for c in df.columns
    if c not in _EXCLUDE and df[c].dtype != object
]

print(f"Features ({len(FEATURE_COLS)}):\n  {FEATURE_COLS}")
print(f"\nTarget : {TARGET}")

# ── 4. Scale features and target ──────────────────────────────────────────────
feat_scaler   = StandardScaler()
target_scaler = StandardScaler()

df[FEATURE_COLS] = feat_scaler.fit_transform(df[FEATURE_COLS])
# Reshape to (N,1) for scaler then flatten back
df[TARGET] = target_scaler.fit_transform(df[[TARGET]]).ravel()

print("\nScaling done. Sample stats after scaling:")
print(df[FEATURE_COLS + [TARGET]].describe().T[["mean", "std", "min", "max"]])


## ARIMA Model (SARIMAX with exogenous features)

ARIMA works on a **single univariate** time series, so we:
- Pick the location with the most data points (or the one set in `ARIMA_LOCATION`)
- Cap the series at `ARIMA_MAX_PTS` rows to keep fitting tractable
- Persist the fitted `SARIMAXResults` object via `pickle` so the model can be reloaded without refitting

> **Why SARIMAX and not plain ARIMA?** SARIMAX accepts exogenous regressors, letting us later attribute each feature's importance via its estimated coefficient.


In [ ]:
# ── Select a single location for the univariate ARIMA series ──────────────────
if ARIMA_LOCATION is None:
    ARIMA_LOCATION = df.groupby("location_id").size().idxmax()
    print(f"Auto-selected location: {ARIMA_LOCATION}")

arima_df = (
    df[df["location_id"] == ARIMA_LOCATION]
    .sort_values("timestamp")
    .reset_index(drop=True)
)

# Optionally cap the series length to keep fitting time reasonable
if ARIMA_MAX_PTS and len(arima_df) > ARIMA_MAX_PTS:
    arima_df = arima_df.iloc[:ARIMA_MAX_PTS].copy()
    print(f"Series capped at {ARIMA_MAX_PTS} rows.")

y_arima    = arima_df[TARGET].values          # 1-D target array (scaled)
exog_arima = arima_df[FEATURE_COLS]           # DataFrame → param names match cols
print(f"ARIMA series length : {len(y_arima)}")

# ── Train SARIMAX or load from checkpoint ─────────────────────────────────────
if ARIMA_CKPT.exists():
    print(f"\nLoading ARIMA checkpoint from: {ARIMA_CKPT}")
    with open(ARIMA_CKPT, "rb") as fh:
        arima_results = pickle.load(fh)
    print("Checkpoint loaded successfully.")
else:
    print(f"\nFitting SARIMAX{ARIMA_ORDER} with {len(FEATURE_COLS)} exogenous features ...")
    # 'trend=n' suppresses an extra intercept term, reducing parameter count
    arima_model = SARIMAX(
        endog=y_arima,
        exog=exog_arima,
        order=ARIMA_ORDER,
        trend="n",
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    arima_results = arima_model.fit(disp=False, maxiter=100)

    # Persist fitted model so we never have to refit
    with open(ARIMA_CKPT, "wb") as fh:
        pickle.dump(arima_results, fh)
    print(f"Checkpoint saved: {ARIMA_CKPT}")
    print(arima_results.summary())

# ── Baseline in-sample metrics ────────────────────────────────────────────────
arima_preds   = arima_results.fittedvalues.values
arima_baseline_mse = mean_squared_error(y_arima, arima_preds)
arima_baseline_mae = mean_absolute_error(y_arima, arima_preds)
print(f"\nARIMA  in-sample  MSE={arima_baseline_mse:.4f}  MAE={arima_baseline_mae:.4f}")


## LSTM Model

### Design choices for the 8 GB shared-VRAM budget
| Technique | Purpose |
|---|---|
| `batch_size = 64` | Limits the live tensor footprint per step |
| `torch.cuda.amp.autocast` + `GradScaler` | Mixed precision (FP16 forward/backward) halves VRAM usage |
| `optimizer.zero_grad(set_to_none=True)` | Frees gradient buffers immediately |
| `torch.cuda.empty_cache()` at checkpoint saves | Returns cached but unused memory to the allocator |
| Gradient clipping (`clip_grad_norm_`) | Prevents exploding gradients, a common LSTM failure mode |

Sequences are built **per location** so windows never span location boundaries.


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# Dataset
# ════════════════════════════════════════════════════════════════════════════════

class ElectricityDataset(Dataset):
    """
    Sliding-window dataset that builds (seq_len × n_features) → scalar sequences
    independently per location, so window boundaries never cross locations.

    All feature arrays are stored as float32 numpy arrays to keep RAM low and
    avoid repeated dtype conversions in __getitem__.
    """

    def __init__(self, df: pd.DataFrame, feature_cols: list, target_col: str, seq_len: int):
        self.seq_len = seq_len
        self.X_arrays: list[np.ndarray] = []   # one (T, F) array per location
        self.y_arrays: list[np.ndarray] = []   # one (T,)  array per location
        self.index: list[tuple] = []            # (array_idx, start_pos) pairs

        for loc_enc in sorted(df["location_id_enc"].unique()):
            loc_df = (
                df[df["location_id_enc"] == loc_enc]
                .sort_values("timestamp")
            )
            X = loc_df[feature_cols].values.astype(np.float32)  # (T, F)
            y = loc_df[target_col].values.astype(np.float32)    # (T,)

            arr_idx = len(self.X_arrays)
            self.X_arrays.append(X)
            self.y_arrays.append(y)

            # Each window: X[i : i+seq_len] → y[i+seq_len]
            for i in range(len(X) - seq_len):
                self.index.append((arr_idx, i))

    def __len__(self) -> int:
        return len(self.index)

    def __getitem__(self, idx: int):
        arr_idx, start = self.index[idx]
        X = self.X_arrays[arr_idx][start : start + self.seq_len]  # (seq_len, F)
        y = self.y_arrays[arr_idx][start + self.seq_len]           # scalar
        return torch.from_numpy(X), torch.tensor(y)


# ════════════════════════════════════════════════════════════════════════════════
# Model
# ════════════════════════════════════════════════════════════════════════════════

class LSTMForecaster(nn.Module):
    """
    Stacked LSTM followed by a single linear projection.
    Input  : (batch, seq_len, n_features)
    Output : (batch, 1)   – point forecast of the next time step's demand
    """

    def __init__(self, n_features: int, hidden_size: int, num_layers: int, dropout: float = 0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            # Dropout is applied between LSTM layers; irrelevant for a single layer
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)        # (B, seq_len, hidden)
        out = out[:, -1, :]          # take the last time-step hidden state
        return self.head(out)        # (B, 1)


print("Dataset and model classes defined.")


In [ ]:
# ── Build dataset & dataloader ────────────────────────────────────────────────
print("Building sliding-window dataset (may take a moment for large files) …")
dataset = ElectricityDataset(df, FEATURE_COLS, TARGET, SEQ_LEN)
print(f"Total sequences : {len(dataset):,}")

# num_workers=0 avoids Windows multiprocessing issues with shared memory
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),  # speeds up host→GPU transfer
)

# ── Instantiate model, optimiser, loss ────────────────────────────────────────
n_features  = len(FEATURE_COLS)
lstm_model  = LSTMForecaster(n_features, HIDDEN_SIZE, NUM_LAYERS, DROPOUT).to(DEVICE)
optimiser   = torch.optim.Adam(lstm_model.parameters(), lr=LR)
criterion   = nn.MSELoss()

# GradScaler: keeps master weights in FP32, runs forward/backward in FP16
# set enabled=False automatically when running on CPU
amp_scaler  = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

start_epoch = 0

# ── Resume from checkpoint if one exists ─────────────────────────────────────
if LSTM_CKPT.exists():
    print(f"\nResuming from checkpoint: {LSTM_CKPT}")
    ckpt = torch.load(LSTM_CKPT, map_location=DEVICE)
    lstm_model.load_state_dict(ckpt["model_state_dict"])
    optimiser.load_state_dict(ckpt["optimizer_state_dict"])
    start_epoch = ckpt["epoch"] + 1   # resume from the next epoch
    print(f"  Resumed at epoch {start_epoch}  (previous loss={ckpt['loss']:.4f})")

# ── Training loop ─────────────────────────────────────────────────────────────
print(f"\nTraining LSTM for {EPOCHS} epoch(s) on {DEVICE} …")
train_losses = []

for epoch in range(start_epoch, EPOCHS):
    lstm_model.train()
    running_loss = 0.0

    for X_batch, y_batch in tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False):
        X_batch = X_batch.to(DEVICE, non_blocking=True)
        # Reshape target to (B, 1) to match the model's (B, 1) output
        y_batch = y_batch.to(DEVICE, non_blocking=True).unsqueeze(1)

        # set_to_none frees the gradient tensors immediately (saves ~VRAM)
        optimiser.zero_grad(set_to_none=True)

        # Mixed-precision forward pass
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
            preds = lstm_model(X_batch)
            loss  = criterion(preds, y_batch)

        # Scaled backward + gradient clipping
        amp_scaler.scale(loss).backward()
        amp_scaler.unscale_(optimiser)
        torch.nn.utils.clip_grad_norm_(lstm_model.parameters(), CLIP_GRAD)
        amp_scaler.step(optimiser)
        amp_scaler.update()

        running_loss += loss.item()

    epoch_loss = running_loss / len(loader)
    train_losses.append(epoch_loss)
    print(f"  Epoch {epoch+1:>3}/{EPOCHS}  loss={epoch_loss:.4f}")

    # ── Checkpoint every SAVE_EVERY epochs and at the final epoch ────────────
    if (epoch + 1) % SAVE_EVERY == 0 or (epoch + 1) == EPOCHS:
        torch.save(
            {
                "epoch"               : epoch,
                "model_state_dict"    : lstm_model.state_dict(),
                "optimizer_state_dict": optimiser.state_dict(),
                "loss"                : epoch_loss,
            },
            LSTM_CKPT,
        )
        # Release cached (but unused) GPU memory back to the allocator
        torch.cuda.empty_cache()
        print(f"    ✓ Checkpoint saved (epoch {epoch+1})")

print("\nLSTM training complete.")

# Plot training curve
plt.figure(figsize=(8, 3))
plt.plot(range(1, len(train_losses) + 1), train_losses, marker="o")
plt.title("LSTM Training Loss (MSE)")
plt.xlabel("Epoch")
plt.ylabel("MSE (scaled)")
plt.tight_layout()
plt.show()


## Permutation Feature Importance

**Algorithm** (same for both models):
1. Compute a baseline MSE with all features intact.
2. For each feature *j* and each of *n_repeats* shuffles:
   - Replace column *j* with a random permutation of its values (breaking the feature–target relationship while keeping the marginal distribution).
   - Re-evaluate MSE.
3. `importance[j] = mean(permuted_MSE − baseline_MSE)` — a higher value means removing the feature's signal hurts more, i.e. the feature is more informative.

### ARIMA-specific note
SARIMAX is a **linear** model in the exogenous regressors: $\hat{y}_t = \text{ARIMA dynamics} + \mathbf{x}_t^\top \boldsymbol{\beta}$.  
We extract $\boldsymbol{\beta}$ directly from `results.params`, so permuting feature *j* changes the contribution by $\Delta_j = (\tilde{x}_{t,j} - x_{t,j})\,\beta_j$ without needing a costly refit.

### LSTM-specific note
Feature arrays stored in `ElectricityDataset.X_arrays` are mutated in-place (then restored) to avoid duplicating memory for large datasets.


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# Permutation importance — ARIMA
# ════════════════════════════════════════════════════════════════════════════════

def permutation_importance_arima(
    results,
    y_true: np.ndarray,
    exog_df: pd.DataFrame,
    feature_cols: list,
    n_repeats: int = 5,
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Analytical permutation importance for a fitted SARIMAX model.

    Because the exogenous part of SARIMAX is linear, we can compute new
    predictions by adjusting only the feature-j contribution:

        ŷ_perm = ŷ_orig  −  x_j · β_j  +  x̃_j · β_j

    This is equivalent to a full permutation but requires no refit and is O(n).

    Returns a DataFrame sorted by importance_mean descending.
    """
    rng = np.random.default_rng(random_state)

    # Extract estimated regression coefficients for each exogenous feature.
    # statsmodels names them after the DataFrame column names when a DataFrame
    # is passed as exog.
    param_names = list(results.param_names)
    beta = {
        col: results.params[param_names.index(col)]
        for col in feature_cols
        if col in param_names
    }

    baseline_preds = results.fittedvalues.values
    baseline_mse   = mean_squared_error(y_true, baseline_preds)

    records = []
    for col in tqdm(feature_cols, desc="ARIMA permutation importance"):
        b = beta.get(col, 0.0)   # coefficient (0 if feature wasn't in the model)
        orig_contrib = exog_df[col].values * b
        deltas = []
        for _ in range(n_repeats):
            perm_contrib = rng.permutation(exog_df[col].values) * b
            # Adjust fitted values by swapping out the original contribution
            perm_preds = baseline_preds - orig_contrib + perm_contrib
            deltas.append(mean_squared_error(y_true, perm_preds) - baseline_mse)
        records.append(
            {"feature": col, "importance_mean": np.mean(deltas), "importance_std": np.std(deltas)}
        )

    return (
        pd.DataFrame(records)
        .sort_values("importance_mean", ascending=False)
        .reset_index(drop=True)
    )


# ════════════════════════════════════════════════════════════════════════════════
# Permutation importance — LSTM
# ════════════════════════════════════════════════════════════════════════════════

def permutation_importance_lstm(
    model: nn.Module,
    dataset: ElectricityDataset,
    feature_cols: list,
    device: torch.device,
    n_repeats: int = 5,
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Model-agnostic permutation importance for the LSTM.

    To stay within the 8 GB VRAM budget:
    - Evaluation runs in batches (no full-dataset tensor materialisation).
    - AMP autocast is applied during inference.
    - Each feature column is mutated in-place inside dataset.X_arrays, then
      restored, so no extra copy of the full feature matrix is ever made.

    Returns a DataFrame sorted by importance_mean descending.
    """
    rng = np.random.default_rng(random_state)
    model.eval()

    # Larger eval batch since no gradients are needed
    eval_loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE * 2,
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
    )

    def _batch_mse() -> float:
        """Compute MSE over the full dataset without storing all predictions."""
        total_sq_err, total_n = 0.0, 0
        with torch.no_grad():
            for X_b, y_b in eval_loader:
                X_b = X_b.to(device, non_blocking=True)
                y_b = y_b.to(device, non_blocking=True).unsqueeze(1)
                with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                    preds = model(X_b)
                    # Sum (not mean) so we can average across the whole dataset
                    total_sq_err += F.mse_loss(preds, y_b, reduction="sum").item()
                total_n += y_b.size(0)
        return total_sq_err / total_n

    baseline_mse = _batch_mse()
    print(f"LSTM baseline MSE : {baseline_mse:.4f}")

    records = []
    for feat_idx, col in enumerate(tqdm(feature_cols, desc="LSTM permutation importance")):
        deltas = []
        for _ in range(n_repeats):
            # Save originals, shuffle in-place across all location arrays
            originals = []
            for arr in dataset.X_arrays:
                col_data = arr[:, feat_idx].copy()
                originals.append(col_data)
                # Overwrite with a permutation (no extra allocation beyond col_data)
                arr[:, feat_idx] = rng.permutation(col_data)

            deltas.append(_batch_mse() - baseline_mse)

            # Restore to leave the dataset unchanged for the next feature
            for arr, orig in zip(dataset.X_arrays, originals):
                arr[:, feat_idx] = orig

        records.append(
            {"feature": col, "importance_mean": np.mean(deltas), "importance_std": np.std(deltas)}
        )

    torch.cuda.empty_cache()
    return (
        pd.DataFrame(records)
        .sort_values("importance_mean", ascending=False)
        .reset_index(drop=True)
    )


print("Permutation importance functions defined.")


In [ ]:
# ── Run ARIMA permutation importance ──────────────────────────────────────────
arima_importance = permutation_importance_arima(
    results      = arima_results,
    y_true       = y_arima,
    exog_df      = exog_arima,        # DataFrame with named columns
    feature_cols = FEATURE_COLS,
    n_repeats    = ARIMA_N_REPEATS,
    random_state = SEED,
)

print("\nARIMA – Permutation Feature Importance (top 20):")
print(arima_importance.head(20).to_string(index=False))


In [ ]:
# ── Run LSTM permutation importance ───────────────────────────────────────────
lstm_importance = permutation_importance_lstm(
    model        = lstm_model,
    dataset      = dataset,
    feature_cols = FEATURE_COLS,
    device       = DEVICE,
    n_repeats    = LSTM_N_REPEATS,
    random_state = SEED,
)

print("\nLSTM – Permutation Feature Importance (top 20):")
print(lstm_importance.head(20).to_string(index=False))


## Results: Visualisation & Combined Rank Table


In [ ]:
# ── Helper: horizontal bar chart with error bars ──────────────────────────────
def _plot_importance(df_imp: pd.DataFrame, title: str, ax, color: str, top_n: int = 20):
    # Take top_n features and sort ascending so the most important appears at top
    top = df_imp.head(top_n).sort_values("importance_mean", ascending=True)
    ax.barh(
        top["feature"],
        top["importance_mean"],
        xerr=top["importance_std"],
        color=color,
        alpha=0.85,
        capsize=3,
        error_kw={"elinewidth": 0.8},
    )
    # Dashed zero line as a reference; negative importance = feature helps noise
    ax.axvline(0, color="black", linewidth=0.7, linestyle="--")
    ax.set_xlabel("Mean ΔMSE when feature is shuffled (scaled units)")
    ax.set_title(title, fontweight="bold", fontsize=11)
    ax.tick_params(axis="y", labelsize=8)


fig, axes = plt.subplots(1, 2, figsize=(18, 8))

_plot_importance(arima_importance, "ARIMA — Top Feature Importances", axes[0], "#4C72B0")
_plot_importance(lstm_importance,  "LSTM  — Top Feature Importances",  axes[1], "#DD8452")

fig.suptitle(
    "Permutation Feature Importance\n(higher = feature is more critical to the model)",
    fontsize=13, y=1.02,
)
plt.tight_layout()
# Save figure alongside the checkpoints for reference
fig.savefig(CHECKPOINT_DIR / "feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved to {CHECKPOINT_DIR / 'feature_importance.png'}")

# ── Combined rank comparison table ────────────────────────────────────────────
arima_ranked = (
    arima_importance[["feature", "importance_mean"]]
    .rename(columns={"importance_mean": "arima_importance"})
    .assign(arima_rank=lambda d: d["arima_importance"].rank(ascending=False).astype(int))
)

lstm_ranked = (
    lstm_importance[["feature", "importance_mean"]]
    .rename(columns={"importance_mean": "lstm_importance"})
    .assign(lstm_rank=lambda d: d["lstm_importance"].rank(ascending=False).astype(int))
)

combined = (
    arima_ranked.merge(lstm_ranked, on="feature")
    .sort_values("arima_rank")
    .assign(rank_diff=lambda d: (d["arima_rank"] - d["lstm_rank"]))  # + means LSTM ranks it lower
    .reset_index(drop=True)
)

print("\nCombined Feature Importance — Rank Comparison (sorted by ARIMA rank):")
print(combined.to_string(index=False))

# ── Persist combined table as CSV ─────────────────────────────────────────────
combined.to_csv(CHECKPOINT_DIR / "feature_importance_ranks.csv", index=False)
print(f"\nRank table saved to {CHECKPOINT_DIR / 'feature_importance_ranks.csv'}")
